In [1]:
# =========================================================================================
# BRONZE INGESTION YELLOW/GREEN TAXI DATA
# =========================================================================================

# Bronze Layer Ingestion for NYC Taxi Data

This notebook ingests greena and yellow taxi data from landing to bronze layer.

## 1. Environment Setup

In [2]:
# Import necessary modules

from utils.spark_session import get_spark_session
from utils.hadoop_setup import complete_hadoop_setup

In [3]:
# Setup Hadoop and create Spark session

complete_hadoop_setup()
spark = get_spark_session()

print(f" Spark session created successfully")
print(f" Spark version: {spark.version}")

✔ HADOOP_HOME set to:, os.environ['HADOOP_HOME']
✔ Added to PATH: C:\hadoop\bin

 ✔ winutils.exe: True
 ✔ hadoop.dll: True

🎉 Setup complete!
 Spark session created successfully
 Spark version: 3.4.1


## 2. Configuration

In [35]:
from pyspark.sql.functions import (
    current_timestamp, current_date, input_file_name, lit, col
)
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from pathlib import Path
from datetime import datetime, timezone
import traceback
import shutil
from typing import Set, List, Optional, Tuple
import time
from urllib.parse import unquote
from pyspark.sql import Row

In [36]:
# Project COnfiguration

PROJECT_ROOT = Path(r"C:\Users\chira\Desktop\data_engineering\PySpark\nyc-taxi-analytics-platform")


# Base paths
LANDING_BASE_PATH = PROJECT_ROOT / "data" / "landing" / "nyc_taxi"
BRONZE_BASE_PATH = PROJECT_ROOT / "data" / "bronze" / "nyc_taxi"
PROCESSED_FILES_PATH = str(PROJECT_ROOT / "data" / "_processed_files")


# Taxi types to process
TAXI_TYPES = ["green", "yellow"]
TARGET_YEAR = "2025"

# Batch ID for auditability
BATCH_ID = f"batch_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"


print(" Configuration loaded:")
print(f"    Landing path: {LANDING_BASE_PATH}")
print(f"    Bronze path: {BRONZE_BASE_PATH}")
print(f"    Tracker path: {PROCESSED_FILES_PATH}")
print(f"    Target year: {TARGET_YEAR}")
print(f"    Batch ID: {BATCH_ID}")

 Configuration loaded:
    Landing path: C:\Users\chira\Desktop\data_engineering\PySpark\nyc-taxi-analytics-platform\data\landing\nyc_taxi
    Bronze path: C:\Users\chira\Desktop\data_engineering\PySpark\nyc-taxi-analytics-platform\data\bronze\nyc_taxi
    Tracker path: C:\Users\chira\Desktop\data_engineering\PySpark\nyc-taxi-analytics-platform\data\_processed_files
    Target year: 2025
    Batch ID: batch_20260317_151600


## 3. Schema Definitions

In [37]:
# Schema for processed files tracker (enhanced with more fields)

PROCESSED_SCHEMA = StructType([
    StructField("file_path", StringType(), False),
    StructField("processed_at", TimestampType(), False),
    StructField("batch_id", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("taxi_type", StringType(), True),
    StructField("status", StringType(), True)
])

## 4. Tracker Manager Class

In [ ]:
class TrackerManager:
    """Manages the processed file tracker Delta table"""

    def __init__(self, spark, processed_files_path: str):
        self.spark = spark
        self.processed_files_path = processed_files_path
        self.processed_uris = set()
        self._initialize_tracker()


    def _initialize_tracker(self) -> None:
        """Initialize tracker table if it doesn't exist"""
        try:
            # Try to read existing tracker
            df = self.spark.read.format("delta").load(self.processed_files_path)
            self.processed_uris = {row.file_path for row in df.collect()
                                   if row.file_path and row.file_path.strip()}
            print(f" Loaded tracker with {len(self.processed_uris):,} entries")
        except Exception as e:
            print(f" Creating new processed files tracker...")
            # Clean up any existing directory
            processed_path_obj = Path(self.processed_files_path)
            if processed_path_obj.exists():
                shutil.rmtree(processed_path_obj)
                print("     Cleaned up existing directory")

            # Create empty tracker
            empty_df = self.spark.createDataFrame([], PROCESSED_SCHEMA)
            empty_df.write.format("delta").mode("overwrite").save(self.processed_files_path)
            self.processed_uris = set()
            print(" Tracker initialized successfully")

    def get_processed_uris(self) -> Set[str]:
        """Get set of processed file URIs"""
        return self.processed_uris.copy()
    
    def refresh(self) -> None:
        """"Refresh the processed URIs from storage"""
        try:
            df = self.spark.read.format("delta").load(self.processed_files_path)
            self.processed_uris = {row.file_path for row in df.collect()
                                   if row.file_path and row.file_path.strip()}
            print(f" Tracker refreshed: {len(self.processed_uris):,} entries")
        except Exception as e:
            print(f" Error refreshing tracker: {e}")

    def mark_as_processed(self, file_uris: Set[str], taxi_type: str,
                          batch_id: str, record_count: int) -> bool:
        """Mark files as processed in the tracker"""
        if not file_uris:
            return True
        
        from pyspark.sql import Row

        # Create rows for tracker with enhanced information
        tracker_rows = []
        for uri in file_uris:
            tracker_rows.append(Row(
                file_path=uri,
                processed_at=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'),
                batch_id=batch_id,
                record_count=record_count,
                taxi_type=taxi_type,
                status="SUCCESS"
            ))
        try:
            tracker_df = self.spark.createDataFrame(tracker_rows, schema=PROCESSED_SCHEMA)
            tracker_df.write.format("delta").mode("append").save(self.processed_files_path)

            # Update local cache
            self.processed_uris.update(file_uris)

            print(f" Marked {len(file_uris)} files as processed")
            return True
        except Exception as e:
            print(f" Failed to update tracker: {e}")
            return False
        
    def remove_from_tracker(self, file_uris: Set[str]) -> bool:
        """Remove files from tracker (for recovery)"""
        if not file_uris:
            return True
        
        try:
            current_df = self.spark.read.format("delta").load(self.processed_files_path)

            # Filter out files to remove
            cleaned_df = current_df.filter(~col("file_path").isin(list(file_uris)))

            # Overwrite tracker
            cleaned_df.write.format("delta").mode("overwrite").save(self.processed_files_path)

            # Refresh cache
            self.refresh()

            print(f" Removed {len(file_uris)} files from tracker")
            return True
        except Exception as e:
            print(f" Failed to remove files fromt tracker: {e}")
            return False
        
    def get_stats(self) -> dict:
        """Get statistics about processed files"""
        try:
            df = self.spark.read.format("delta").load(self.processed_files_path)


            # Get the grouped data and convert to dict
            grouped = df.groupBy("taxi_type").count().collect()
            by_taxi_type = [{"taxi_type": row["taxi_type"], "count": row["count"]} for row in grouped]
            stats = {
                "total_files": df.count(),
                "by_taxi_type": by_taxi_type,
                "latest_run": df.agg({"processed_at": "max"}).collect()[0][0]
            }
            return stats
        except Exception as e:
            return {"error": str(e)}

## 5. File Discovery Class

In [39]:
class FileDiscovery:
    """Discovers and validates files in landing zone"""

    def __init__(self, base_path: Path, target_year: str):
        self.base_path = base_path
        self.target_year = target_year

    def to_file_uri(self, path: Path) -> str:
        """Convert path to canonical file URI"""
        return unquote(path.resolve().as_uri())
    
    def discover_files(self, taxi_type: str) -> Tuple[List[Path], Set[str], Path]:
        """
        Discover valid parquet files for a taxi_type
        Returns: (valid_files, file_uris, year_path)
        """
        taxi_path = self.base_path / taxi_path
        year_path = taxi_path / self.target_year

        if not year_path.exists():
            raise FileNotFoundError(f" Taxi path not found: {taxi_path}")
        
        if not year_path.exists():
            print(f" Year path not found: {year_path}")
            return [], set(), year_path
        
        # Find all parquet files
        parquet_files = list(year_path.rglob("*.parquet"))

        # Filter valid (non-empty) files
        valid_files = []
        invalid_files = []

        for f in parquet_files:
            try:
                size = f.stat().st_size
                if size > 0:
                    valid_files.append(f)
                else:
                    invalid_files.append(f)
            except Exception as e:
                print(f" Error checking file {f}: {e}")

        # Log findings
        print(f"  File discovery for {taxi_type}:")
        print(f"    Total parquet files: {len(parquet_files)}")
        print(f"    Valid files: {len(valid_files)}")

        if invalid_files:
            print(f" Empty/invalid files: {len(invalid_files)}")

        # Show valid files with sizes
        for f in valid_files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f" {f.name}: {size_mb:.2f} MB")


        # Generate URIs
        file_uris = {self.to_file_uri(f) for f in valid_files}

        return valid_files, file_uris, year_path
    

## 6. Bronze Ingestion Function (Robust Version)

In [40]:
def ingest_to_bronze_robust(taxi_type: str, tracker_manager: TrackerManager) -> Tuple[bool, dict]:
    """
    Robust version of bronze ingestion with validation and error handling
    Returns: (success, metrics)
    """
    start_time = time.time()

    # Initialize metrics
    metrics = {
        "taxi_type": taxi_type,
        "batch_id": BATCH_ID,
        "status": "Failed",
        "files_found": 0,
        "new_files": 0,
        "records_ingested": 0,
        "bronze_path": str(BRONZE_BASE_PATH / taxi_type),
        "error": None
    }

    landing_path = LANDING_BASE_PATH / taxi_type
    bronze_path = BRONZE_BASE_PATH / taxi_type

    print(f"\n{'='*70}")
    print(f" PROCESSING {taxi_type.upper()} TAXI DATA")
    print(f"{'='*70}")
    print(f"    Landing: {landing_path}")
    print(f"    Bronze: {bronze_path}")
    print(f"    Batch ID: {BATCH_ID}")

    # Check if landing path exists
    if not landing_path.exists():
        msg = f" No landing data found for {taxi_type} at: {landing_path}"
        print(msg)
        metrics["error_message"] = msg
        metrics["status"] = "FAILED_NO_LANDING"
        return False, metrics
    
    # Check year directory
    year_path = landing_path / TARGET_YEAR
    if not year_path.exists():
        msg = f" No {TARGET_YEAR} directory found for {taxi_type}"
        print(msg)
        metrics["error_message"] = msg
        metrics["status"] = "FAILED_NO_YEAR_DIR"
        return False, metrics
    
    # Discover files
    file_discovery = FileDiscovery(LANDING_BASE_PATH, TARGET_YEAR)

    try:
        valid_files, file_uris, _ = file_discovery.discover_files(taxi_type)
        metrics["files_found"] = len(valid_files)
    except Exception as e:
        msg = f" Error discovering files: {str(e)}"
        print(msg)
        metrics["error_message"] = msg
        metrics["status"] = "FAILED_DISCOVERY"
        return False, metrics
    
    if not valid_files:
        msg = f" No valid parquet files found for {taxi_type}"
        print(msg)
        metrics["status"] = "SUCCESS_NO_DATA"
        metrics["error_message"] = msg
        return True, metrics
    
    # Get processed files from tracker
    tracker_manager.refresh()
    processed_uris = tracker_manager.get_processed_uris()

    # Find new files
    new_files_uris = file_uris - processed_uris
    metrics["new_files"] = len(new_files_uris)

    # Check if bronze data exists for files marked as processed
    if not new_files_uris:
        print(f" No new files to ingest for {taxi_type}")

        # Validate that bronze data actually exists
        if bronze_path.exists():
            try:
                bronze_df = spark.read.format("delta").load(str(bronze_path))
                bronze_count = bronze_df.count()
                print(f"    Bronze data verified: {bronze_count:,} records")
                metrics["status"] = "SUCCESS_UP_TO_DATE"
                metrics["records_ingested"] = bronze_count
                return True, metrics
            except Exception as e:
                print(f"    Bronze path exists but cannot be read: {e}")
                print(f"    Re-ingesting files...")
                new_files_uris = file_uris # Force re-ingestion
                metrics["new_files"] = len(new_files_uris)

        if not new_files_uris:
            metrics["status"] = "SUCCESS_UP_TO_DATE"
            return True, metrics
        
        print(f" New files to process: {len(new_files_uris)}")
        for uri in sorted(new_files_uris):
            print(f"    {Path(uri).name}")

    # Process new files
    new_file_paths = [str(f) for f in valid_files
                      if file_discovery.to_file_uri(f) in new_files_uris]
    
    year_path_str = str(year_path).replace("\\", "/")

    try:
        # Read and process files
        print(f"    Reading {len(new_file_paths)} file(s)...")

        # Read files with schema merging
        df = (
            spark.read
                 .option("basePath", year_path_str)
                 .option("mergeSchema", "true")
                 .parquet(*new_file_paths)
        )

        # Add audit columns
        df_bronze = (
            df
            .withColumn("_source_file", input_file_name())
            .withColumn("_ingestion_date", current_date())
            .withColumn("_ingestion_timestamp", current_timestamp())
            .withColumn("_batch_id", lit(BATCH_ID))
            .withColumn("_taxi_type", lit(taxi_type))
            .withColumn("_pipeline_version", lit("2.0.0"))
        )

        # Get record count
        new_count = df_bronze.count()
        metrics["records_ingested"] = new_count()

        print(f" Loaded {new_count:,} new records")

        # Show sample of data
        print("\n Sample of ingested data:")
        df_bronze.select(
            "taxi_type", "_source_file", "_ingestion_timestamp", "_batch_id"
        ).show(5, truncate=False)

        # Write to bronze
        print(f"\n Writing to bronze layer...")
        bronze_path.mkdir(parents=True, exist_ok=True)

        df_bronze.write \
            .format("delta") \
            .mode("append") \
            .partitionBy("_ingestion_date") \
            .option("overwriteSchema", "false") \
            .save(str(bronze_path))
        
        print(f" Successfully wrote to bronze layer")

        # Verify write
        print(f"\n Verifying write...")
        time.sleep(1) # Brief pause for write to complete

        verification_df = spark.read.format("delta").load(str(bronze_path))
        verification_count = verification_df.count()

        print(f"    Excepted records: {new_count:,}")
        print(f"    Actual records: {verification_count:,}")

        if verification_count >= new_count:
            print(f" Write verified successfully")

            # Update tracker
            print(f"\n Updating tracker...")
            if tracker_manager.mark_as_processed(
                new_files_uris, taxi_type, BATCH_ID, new_count
            ):
                print(f" Tracker updated successfully")

                # Final success message
                elapsed_time = time.time() - start_time()
                print("\n{'='*70}")
                print(f" SUCCESS: {taxi_type.upper()} ingestion completed")
                print(f"{'='*70}")
                print(f"    Records ingested: {new_count:,}")
                print(f"    New files: {len(new_files_uris)}")
                print(f"    Time: {elapsed_time:.2f} seconds")
                print(f"    Bronze path: {bronze_path}")

                metrics["status"] = "SUCCESS"
                metrics["elapsed_time"] = elapsed_time
                return True, metrics
            else:
                raise Exception("Failed to update tracker")
        else:
            raise Exception(
                f"Write verification failed: Expected {new_count}, got {verification_count}"
            )
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        error_msg = str(e)
        print(f"\n Error during ingestion: {error_msg}")
        traceback.print_exc()

        # Clean up partial write if it happened
        if bronze_path.exists():
            print(f" Partial write may exist. Manual cleanup may be needed.")

        metrics["status"] = "FAILED"
        metrics["error_message"] = error_msg
        metrics["elapsed_time"] = elapsed_time
        return False, metrics
    

## 7. Convenience Wrapper Functions

In [41]:
def ingest_green_to_bronze(tracker_manager: TrackerManager) -> Tuple[bool, dict]:
    """Convenience function to ingest green taxi data"""
    return ingest_to_bronze_robust("green", tracker_manager)

def ingest_yellow_to_bronze(tracker_manager: TrackerManager) -> Tuple[bool, dict]:
    """Convenience function to ingest yellow taxi data"""
    return ingest_to_bronze_robust("yellow", tracker_manager)

# 8. Initialize Tracker

In [42]:
# Initialize tracker manager
tracker_manager = TrackerManager(spark, PROCESSED_FILES_PATH)

# Show tracker stats
print("\n Tracker Statistics:")
stats = tracker_manager.get_stats()
if "error" not in stats:
    print(f"    Total files processed: {stats['total_files']:,}")
    print(f"    Latest run: {stats['latest_run']}")
    print(f"\n    Breakdown by taxi_type:")
    for row in stats['by_taxi_type']:
        print(f"    - {row['taxi_type']}: {row['count']} files")
else:
    print(f"    {stats['error']}")

 Loaded tracker with 2 entries

 Tracker Statistics:
    [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `taxi_type` cannot be resolved. Did you mean one of the following? [`file_path`, `processed_at`].;
'Aggregate ['taxi_type], ['taxi_type, count(1) AS count#1018L]
+- Relation [file_path#1011,processed_at#1012] parquet



In [43]:
# Debug: See what columns exist
tracker_df = spark.read.format("delta").load(PROCESSED_FILES_PATH)
print("Columns:", tracker_df.columns)
print("Schema:")
tracker_df.printSchema()
print("Sample data:")
tracker_df.show(5)

Columns: ['file_path', 'processed_at']
Schema:
root
 |-- file_path: string (nullable = true)
 |-- processed_at: timestamp (nullable = true)

Sample data:
+--------------------+--------------------+
|           file_path|        processed_at|
+--------------------+--------------------+
|file:///C:/Users/...|2026-02-20 14:24:...|
|file:///C:/Users/...|2026-03-11 19:46:...|
+--------------------+--------------------+

